# Пятый этап: RAG-ответы с GigaChat-2

Ноутбук показывает зафиксированные SciFact-примеры, retrieval-контекст, точный prompt, сохранённые ответы GigaChat и их оценку. Он намеренно **не отправляет новые запросы на генерацию**: восемь ответов уже получены двумя подтверждёнными пакетами по четыре.

In [1]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

# Находим корень проекта независимо от каталога запуска Jupyter.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден pyproject.toml')
os.chdir(PROJECT_ROOT)

def read_jsonl(path: Path) -> list[dict]:
    """Прочитать сохранённые артефакты без повторных API-вызовов."""
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

print(f'Корень: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')

Корень: /Users/e.ilyina/Desktop/Портфолио/RAG
Python: 3.13.0


## 1. Зафиксированные примеры

Выбор сделан до генерации: seed 42, четыре запроса с одним релевантным документом и четыре с несколькими.

In [2]:
plan_path = PROJECT_ROOT / 'data/experiments/stage_5_queries.json'
plan = json.loads(plan_path.read_text(encoding='utf-8'))
print('Метод:', plan['selection_method'], '| seed:', plan['random_seed'])
for row in plan['examples']:
    print(f"{row['query_id']:>4} | relevant={len(row['relevant_doc_ids'])} | {row['query_text']}")

Метод: stratified_random_without_replacement | seed: 42
1271 | relevant=1 | The severity of cardiac involvement in amyloidosis can be described by the degree of transmurality of late gadolinium enhancement in MRI.
1089 | relevant=1 | Smc5/6 engagment drives the activation of SUMO E3 ligase Mms21 by ATP-dependent remolding.
 386 | relevant=1 | Errors in peripheral IV drug administration are most common during bolus administration and multiple-step medicine preparations.
 295 | relevant=1 | Crosstalk between dendritic cells (DCs) and innate lymphoid cells (ILCs) is important in the regulation of intestinal homeostasis.
 179 | relevant=4 | Birth-weight is positively associated with breast cancer.
 133 | relevant=5 | Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.
1274 | relevant=3 | The tip of the inner tube of the toxic type VI secretion system (T6SS) antibacterial effector in Esch

## 2. Retrieval-контексты

Для каждого запроса сохранены четыре лучших recursive-чанка. Проверяем, попал ли в контекст хотя бы один документ из qrels.

In [3]:
prepared_path = PROJECT_ROOT / 'reports/results/rag/stage_5_prepared.jsonl'
prepared = read_jsonl(prepared_path)
print('query | sources | prompt chars | relevant document in context')
for row in prepared:
    print(f"{row['query_id']:>5} | {len(row['sources']):>7} | {len(row['prompt']):>12} | {bool(row['retrieved_relevant_doc_ids'])}")

query | sources | prompt chars | relevant document in context
 1271 |       4 |         4147 | True
 1089 |       4 |         4645 | True
  386 |       4 |         5590 | True
  295 |       4 |         4384 | False
  179 |       4 |         5066 | True
  133 |       4 |         4979 | True
 1274 |       4 |         4893 | True
  967 |       4 |         4623 | True


## 3. Точный prompt

Показываем prompt первого примера именно в том виде, в котором он был отправлен модели. Credentials в артефакты не записываются.

In [4]:
print(prepared[0]['prompt'])

Ты — аккуратный научный ассистент. Проверь утверждение, используя ТОЛЬКО
предоставленные ниже источники. Не используй сведения из памяти и не выдумывай факты.

Правила ответа:
1. Пиши по-русски и кратко.
2. Первая строка должна иметь один из видов:
   ВЫВОД: ПОДТВЕРЖДАЕТСЯ
   ВЫВОД: ОПРОВЕРГАЕТСЯ
   ВЫВОД: НЕДОСТАТОЧНО ДАННЫХ
3. Затем дай объяснение и поставь ссылки [1], [2] только после утверждений,
   которые действительно поддерживаются соответствующим источником.
4. Не ссылайся на номера, которых нет в контексте.
5. В конце напиши: ИСТОЧНИКИ: и перечисли использованные номера.

НАУЧНОЕ УТВЕРЖДЕНИЕ:
The severity of cardiac involvement in amyloidosis can be described by the degree of transmurality of late gadolinium enhancement in MRI.

ИСТОЧНИКИ:
[1] doc_id=13768432; chunk_id=13768432:recursive:0002
TITLE: Prognostic Value of Late Gadolinium Enhancement Cardiovascular Magnetic Resonance in Cardiac Amyloidosis
TEXT: ##ricular mass index ( hazard ratio, 4. 1 ; 95 % confidence interval

## 4. Как выполнялась генерация

Команды ниже только отображаются. Повторный запуск заблокирован защитой от дубликатов, а один batch физически ограничен пятью запросами.

In [5]:
# Эти строки документируют два разрешённых запуска, но subprocess здесь не вызывается.
commands = [
    'rag-playground rag-generate --query-id 1271 --query-id 1089 --query-id 386 --query-id 295 --mode rag --confirm-generation',
    'rag-playground rag-generate --query-id 179 --query-id 133 --query-id 1274 --query-id 967 --mode rag --confirm-generation',
]
print(*commands, sep='\n')

rag-playground rag-generate --query-id 1271 --query-id 1089 --query-id 386 --query-id 295 --mode rag --confirm-generation
rag-playground rag-generate --query-id 179 --query-id 133 --query-id 1274 --query-id 967 --mode rag --confirm-generation


## 5. Ответы и автоматические проверки

Печатаем сохранённые ответы, token usage и результаты проверки номеров источников.

In [6]:
answers_path = PROJECT_ROOT / 'reports/results/rag/stage_5_answers.jsonl'
answers = read_jsonl(answers_path)
for row in answers:
    print(f"\n{'=' * 80}\nQUERY {row['query_id']} | total tokens={row['total_tokens']} | invalid citations={row['invalid_source_numbers']}")
    print(row['answer'])


QUERY 1271 | total tokens=1271 | invalid citations=[]
ВЫВОД: ПОДТВЕРЖДАЕТСЯ

Утверждение подтверждается источником [1]. Согласно этому источнику, тяжесть кардиоваскулярного вовлечения при амилоидозе можно охарактеризовать степенью трансмуральности позднего усиления гадолиния (LGE) при МРТ. Этот метод позволяет надежно определять степень поражения сердца и представляет собой признак продвинутого состояния заболевания.

QUERY 1089 | total tokens=1309 | invalid citations=[]
ВЫВОД: ПОДТВЕРЖДАЕТСЯ

Утверждение подтверждается источниками [1] и [2]. Согласно источнику [1], взаимодействие комплекса SMC5/6 с АТФ позволяет физически ремоделировать молекулу, что способствует активации SUMO E3 лигазы Mms21. Источник [2] подтверждает роль комплекса SMC5/6 в регуляции длины теломер через SUMO-модификацию белков, участвующих в альтернативном удлинении теломер (ALT). Оба источника указывают на связь между взаимодействием SMC5/6 с АТФ и активацией SUMO-зависимых процессов. 

ИСТОЧНИКИ: [1], [2]

QUERY

In [7]:
summary_path = PROJECT_ROOT / 'reports/results/rag/stage_5_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "answers": 8,
  "answers_with_citations": 8,
  "answers_with_invalid_citations": 0,
  "baseline_answers": 0,
  "mean_generation_latency_ms": 1240.454,
  "mean_total_tokens": 1326.25,
  "rag_answers": 8,
  "rag_contexts_with_relevant_document": 7,
  "total_completion_tokens": 1008,
  "total_prompt_tokens": 9602
}


## 6. Ручная оценка

Шкала 0–2 отдельно оценивает правильность, опору на контекст, полноту и качество цитат. Это не LLM-as-a-judge: оценки выставлены после сопоставления ответа с сохранёнными источниками.

In [8]:
manual_path = PROJECT_ROOT / 'data/experiments/stage_5_manual_evaluation.csv'
with manual_path.open(encoding='utf-8') as source:
    manual_rows = list(csv.DictReader(source))
score_fields = ('correctness_0_2', 'groundedness_0_2', 'completeness_0_2', 'citation_quality_0_2')
for row in manual_rows:
    scores = ', '.join(f"{field[:-4]}={row[field]}" for field in score_fields)
    print(f"{row['query_id']}: {scores}; error={row['error_type']}")

1271: correctness=2, groundedness=2, completeness=2, citation_quality=1; error=format_noncompliance
1089: correctness=2, groundedness=1, completeness=2, citation_quality=1; error=generation_overclaim
386: correctness=2, groundedness=2, completeness=2, citation_quality=2; error=none
295: correctness=2, groundedness=2, completeness=2, citation_quality=1; error=retrieval_qrels_miss
179: correctness=2, groundedness=2, completeness=2, citation_quality=2; error=none
133: correctness=2, groundedness=2, completeness=2, citation_quality=2; error=none
1274: correctness=2, groundedness=2, completeness=2, citation_quality=2; error=none
967: correctness=2, groundedness=2, completeness=2, citation_quality=1; error=format_noncompliance


## 7. Проверки проекта

Ruff и unit-тесты не отправляют запросы GigaChat: SDK подменяется fake-клиентом.

In [9]:
ruff = subprocess.run(
    [sys.executable, '-m', 'ruff', 'check', 'src', 'tests'],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
)
tests = subprocess.run(
    [sys.executable, '-m', 'pytest'],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True,
)
print('Ruff:', ruff.stdout.strip())
print('Pytest:', tests.stdout.strip())

Ruff: All checks passed!
Pytest: sss.................................................                     [100%]
49 passed, 3 skipped in 3.15s


## Вывод

GigaChat-2 сформировал восемь ответов: все содержат валидные номера источников, но анализ обнаружил два нарушения формата и одно чрезмерное обобщение дополнительного источника. Запрос 295 демонстрирует отдельную retrieval-проблему: документ из qrels находился только на девятой позиции.